# 03 · Review completed evidence

**Question:** What did the completed experiment establish, and what should we test next?

This notebook checks saved artifacts and recalculates metrics from out-of-fold predictions. It never trains a model. Open this notebook after restoring a saved run; you do not need to repeat notebook 02 when reviewing historical results.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML, FileLink
from jigsaw_rules.data import load_data, audit
from jigsaw_rules.runtime import Progress, environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = ["#087f8c", "#bd633b", "#334ea0", "#70923b"]
display(HTML("<div style='padding:18px;background:#edf6f5;border-left:5px solid #087f8c'>"
             "<b>Jigsaw research workspace</b><br>Every result must identify its data and validation protocol.</div>"))
print("Project:", root)


In [ ]:
from jigsaw_rules.review import review_run
is_demo = (root / 'data/raw/SYNTHETIC.txt').exists()
evidence = review_run(root, os.environ.get('JIGSAW_RUN_ID'), allow_synthetic=is_demo)
print('Data kind:', evidence['data_kind'])
print('Run:', evidence['run_id'])
print('Training rows:', evidence['training_rows'])
print('Training SHA-256:', evidence['training_sha256'])
if evidence['data_kind'] == 'synthetic':
    display(HTML('<b>SYNTHETIC SOFTWARE TEST — not competition performance</b>'))

## The generalization gap
Compare the two validation protocols separately. A small difference between models is not evidence of a statistically reliable improvement. With only two rules, new-rule transfer remains weakly measured. AUC evaluates ranking; probability quality needs log loss, Brier score, and calibration diagnostics as well.

In [ ]:
records = evidence['results']
summary = pd.DataFrame([{'Model': r['model'], 'Protocol': r['protocol'], **{k: v for k, v in r['metrics'].items() if isinstance(v, float)}} for r in records])
display(summary.round(4))
fig = px.bar(summary, x='Protocol', y='rule_macro_auc', color='Model', barmode='group', title='Saved out-of-fold evidence: familiar versus held-out rules')
fig.update_yaxes(range=[0, 1])
fig.add_hline(y=0.5, line_dash='dash', annotation_text='Chance ranking')
fig.show()
display(FileLink(str(root / 'reports/private/report.html')))
display(FileLink(str(root / 'reports/private/results.json')))

## Phase 2 decision
The next controlled experiment compares a frozen semantic embedding/example matcher with a rule-conditioned cross-encoder under the same saved splits. Start from a pinned model revision, preserve embedding batches, record latency and peak memory, and compare per-rule as well as aggregate performance.

See `docs/PHASE_2.md` for the predeclared experiment and hardware gate. This review does not download weights or launch a paid job. Historical results remain valid evidence for their recorded source version; a code update does not require retraining merely to view them.